# z610 - Features Cliente-Producto (grano nuevo)
Campo clave: (customer_id, product_id). Usa `sell-in-zeroes.txt` SIN agregar por producto -- ya viene al grano correcto. ~18-19M filas, ~735k series. Puede tardar varios minutos.

In [1]:
!pip install -q polars pyarrow

In [2]:
import os
import numpy as np
import polars as pl
import warnings
warnings.filterwarnings("ignore")

In [3]:
PARAM = {
    'experimento': 'CP601',
    'sellin_zeroes_path': '/home/ds/datasets/sell-in-zeroes.txt',
    'windows': [3, 6, 9, 12, 18, 24, 36],
    'lags': [1, 2, 3, 6, 12],
    'umbral_joven_meses': 12
}

ruta = os.path.join('/home/ds/exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/CP601


## 1. Cargar (grano customer_id x product_id x periodo, sin agregar)

In [4]:
df = pl.read_csv(PARAM['sellin_zeroes_path'], separator=",").select(
    ["customer_id", "product_id", "periodo", "tn"]
)
df = df.sort(["customer_id", "product_id", "periodo"])
print(df.shape)

(16648066, 4)


## 2. Lags

In [5]:
CLAVE = ["customer_id", "product_id"]

lag_exprs = [
    pl.col("tn").shift(n).over(CLAVE).alias(f"tn_lag_{n}")
    for n in PARAM['lags']
]
df = df.with_columns(lag_exprs)

## 3. Rolling (mismo criterio: shift(1) antes del rolling, nunca incluye el periodo actual)

In [6]:
df = df.with_columns(
    pl.col("tn").shift(1).over(CLAVE).alias("tn_shift1")
)

rolling_exprs = []
for w in PARAM['windows']:
    base = pl.col("tn_shift1")
    rolling_exprs += [
        base.rolling_mean(window_size=w, min_periods=1).over(CLAVE).alias(f"tn_media_{w}"),
        base.rolling_std(window_size=w, min_periods=2).over(CLAVE).pow(2).alias(f"tn_var_{w}"),
        base.rolling_max(window_size=w, min_periods=1).over(CLAVE).alias(f"tn_max_{w}"),
        base.rolling_min(window_size=w, min_periods=1).over(CLAVE).alias(f"tn_min_{w}"),
        base.rolling_sum(window_size=w, min_periods=1).over(CLAVE).alias(f"tn_suma_{w}"),
    ]
df = df.with_columns(rolling_exprs)

df = df.with_columns(
    (pl.col("tn_suma_3") / (pl.col("tn_media_12") * 3 + 1e-6)).alias("ratio_sobrecompra_3_12")
)

## 4. Edad del producto (nacimiento GLOBAL del producto, no del par cliente-producto)

In [7]:
def periodo_a_meses(periodo: int) -> int:
    return (periodo // 100) * 12 + (periodo % 100)

df = df.with_columns(
    pl.col("periodo").map_elements(periodo_a_meses, return_dtype=pl.Int64).alias("periodo_m")
)

# nacimiento del producto: primer periodo con venta de CUALQUIER cliente
ventas_reales = df.filter(pl.col("tn") > 0)
nacimiento_producto = ventas_reales.group_by("product_id").agg(
    pl.col("periodo_m").min().alias("nacimiento_m")
)
df = df.join(nacimiento_producto, on="product_id", how="left")

df = df.with_columns(
    (pl.col("periodo_m") - pl.col("nacimiento_m")).alias("edad_producto")
)
df = df.with_columns(
    (pl.col("edad_producto") < PARAM['umbral_joven_meses']).cast(pl.Int8).alias("producto_joven")
)

## 5. Patron de frecuencia de compra (por PAR cliente-producto)
~735k grupos -- puede tardar varios minutos. Cada valor en la fila `i` usa SOLO datos hasta `i-1`.

In [8]:
def features_frecuencia(g: pl.DataFrame) -> pl.DataFrame:
    tn = g["tn"].to_numpy()
    periodo_m = g["periodo_m"].to_numpy()
    n = len(tn)

    meses_desde_ultima_compra = np.full(n, np.nan)
    racha_actual_sin_compra = np.full(n, np.nan)
    mayor_racha_historica = np.full(n, np.nan)
    promedio_gap_historico = np.full(n, np.nan)
    tn_promedio_por_compra_hist = np.full(n, np.nan)

    ultima_compra_idx = None
    racha_actual = 0
    max_racha = 0
    gaps = []
    tn_compras = []

    for i in range(n):
        if ultima_compra_idx is not None:
            meses_desde_ultima_compra[i] = periodo_m[i] - periodo_m[ultima_compra_idx]
        racha_actual_sin_compra[i] = racha_actual
        mayor_racha_historica[i] = max_racha
        if gaps:
            promedio_gap_historico[i] = float(np.mean(gaps))
        if tn_compras:
            tn_promedio_por_compra_hist[i] = float(np.mean(tn_compras))

        if tn[i] > 0:
            if ultima_compra_idx is not None:
                gap = periodo_m[i] - periodo_m[ultima_compra_idx] - 1
                gaps.append(gap)
            tn_compras.append(tn[i])
            ultima_compra_idx = i
            racha_actual = 0
        else:
            racha_actual += 1
            max_racha = max(max_racha, racha_actual)

    return g.with_columns([
        pl.Series("meses_desde_ultima_compra", meses_desde_ultima_compra),
        pl.Series("racha_actual_sin_compra", racha_actual_sin_compra),
        pl.Series("mayor_racha_historica", mayor_racha_historica),
        pl.Series("promedio_gap_historico", promedio_gap_historico),
        pl.Series("tn_promedio_por_compra_hist", tn_promedio_por_compra_hist),
    ])

df = df.sort(CLAVE + ["periodo"])
df = df.group_by(CLAVE, maintain_order=True).map_groups(features_frecuencia)

## 6. Guardar

In [9]:
salida = os.path.join(ruta, "tb_features_CP601.parquet")
df.write_parquet(salida)
print(salida)
print(df.shape)

/home/ds/exp/CP601/tb_features_CP601.parquet
(16648066, 55)
